# CPU Overheads

In [1]:
import os
import re
import json
from common import *
from collections import defaultdict
from data import RawDataFile
from experiment import Treatment, NetworkSetting
from treatments.picoquic import treatment_map as http_treatment_map, NETWORK_SETTING as picoquic_ns
from treatments.media import treatment_map as media_treatment_map, NETWORK_SETTING as media_ns

## Collect data

```
cd deps
./build_deps.sh [7|9]
cd ..
sudo -E python3 emulation/main.py --bw1 20 --bw2 20 --delay2 100 --loss1 4 --loss2 0 \
--proxy sidekick --freq-pkts 16 --freq-ms 10 --threshold 160 --riblt --quack-hint \
picoquic --client-quacker --ack-delay 30 -n 25000000 
```

In [2]:
# Make the data directory
home_base = f'{DATA_HOME}/cycles_base/'
home_quack = f'{DATA_HOME}/cycles_quack/'
os.system(f'mkdir -p {home_base}')
os.system(f'mkdir -p {home_quack}')

0

In [11]:
# HTTP benchmark
def gen_http_cmd(label, time_s, base: bool):
    if base:
        print('./deps/build_deps.sh 7')
        home = home_base
    else:
        print('./deps/build_deps.sh 9')
        home = home_quack
    treatment = http_treatment_map(label)
    data_size = 125000 * 20 * time_s # 125000 * bottleneck_bw * time_s
    cmd = RawDataFile(treatment, picoquic_ns, '').cmd(data_size=data_size, num_trials=1, logdir=home, timeout=time_s)
    print(cmd)

gen_http_cmd('picoquic_iblt_30ms_hint', 60, base=True)
gen_http_cmd('picoquic_iblt_30ms_hint', 60, base=False)

./deps/build_deps.sh 7
sudo -E python3 emulation/main.py --timeout 60 --bw1 50 --bw2 20 --delay1 2 --delay2 30 --loss1 4 --loss2 0 -t 1 --logdir /home/ubuntu/sidekick-downloads/data/cycles_base/ --label picoquic_iblt_30ms_hint --proxy sidekick --freq-pkts 16 --freq-ms 10 --threshold 160 --riblt --quack-hint --network-statistics picoquic --client-quacker --ack-delay 30 -n 150000000
./deps/build_deps.sh 9
sudo -E python3 emulation/main.py --timeout 60 --bw1 50 --bw2 20 --delay1 2 --delay2 30 --loss1 4 --loss2 0 -t 1 --logdir /home/ubuntu/sidekick-downloads/data/cycles_quack/ --label picoquic_iblt_30ms_hint --proxy sidekick --freq-pkts 16 --freq-ms 10 --threshold 160 --riblt --quack-hint --network-statistics picoquic --client-quacker --ack-delay 30 -n 150000000


In [6]:
# Media benchmark
def gen_media_cmd(label, time_s, base: bool):
    if base:
        print('./deps/build_deps.sh 7')
        home = home_base
    else:
        print('./deps/build_deps.sh 9')
        home = home_quack
    treatment = media_treatment_map(label)
    return RawDataFile(treatment, media_ns, '').cmd(duration=time_s, logdir=home)

print(gen_media_cmd('iblt_delay45_hint_nack', 30, True))
print(gen_media_cmd('iblt_delay45_hint_nack', 30, False))

./deps/build_deps.sh 7
sudo -E python3 emulation/main.py --bw1 50 --bw2 20 --delay1 10 --delay2 30 --loss1 10 --loss2 0 --logdir /home/ubuntu/sidekick-downloads/data/cycles_base/ --label iblt_delay45_hint_nack --proxy sidekick --freq-ms 0 --threshold 32 --riblt --quack-hint --quack-nack --freq-pkts 0 --network-statistics media --client-quacker --ack-delay 45 --duration 30
./deps/build_deps.sh 9
sudo -E python3 emulation/main.py --bw1 50 --bw2 20 --delay1 10 --delay2 30 --loss1 10 --loss2 0 --logdir /home/ubuntu/sidekick-downloads/data/cycles_quack/ --label iblt_delay45_hint_nack --proxy sidekick --freq-ms 0 --threshold 32 --riblt --quack-hint --quack-nack --freq-pkts 0 --network-statistics media --client-quacker --ack-delay 45 --duration 30


## Cycles

In [12]:
def print_router_log(data_home):
    filename = f'{data_home}/router.log'
    with open(filename, 'r') as f:
        print(f.read())

print_router_log(home_base)
print_router_log(home_quack)

Creating logfile /home/ubuntu/sidekick-downloads/data/cycles_base//router.log
Ready to start Sidekick with client p1-eth0; expecting server p1-eth1
INFO [proxy] Used cset to isolate CPU 3
INFO [proxy] Set CPU affinity for current process: 3
INFO [proxy::sidekick::base] Instant { tv_sec: 1324580, tv_nsec: 365529267 } Received discovery packet from client. Sidekick: ac10010aa1faac10010b1484, Base: ac10020a1151ac10010ad7c0. Update: false. riblt=true offset=21 threshold=160 cache_policy=Optimistic
INFO [proxy::sidekick::base] Instant { tv_sec: 1324580, tv_nsec: 365568563 } Received discovery packet from client. Sidekick: ac10010aa1faac10010b1484, Base: ac10020a1151ac10010ad7c0. Update: true. riblt=true offset=21 threshold=160 cache_policy=Optimistic
INFO [proxy::sidekick::base] Instant { tv_sec: 1324580, tv_nsec: 365578390 } Received discovery packet from client. Sidekick: ac10010aa1faac10010b1484, Base: ac10020a1151ac10010ad7c0. Update: true. riblt=true offset=21 threshold=160 cache_polic